In [1]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 07 — COUNTERFACTUAL ANALYSIS
# ============================================================
#
# Purpose:
# Evaluate explicitly defined counterfactual recovery scenarios
# using the validated account-level analytical dataset.
#
# IMPORTANT:
# - Counterfactuals are scenario estimates, NOT observed effects.
# - No causal claim is made.
# - Raw source data is never modified.
# - Assumptions are explicitly recorded.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

INPUT_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("=" * 90)
print("CREDRESOLVE — COUNTERFACTUAL ANALYSIS")
print("=" * 90)


# ============================================================
# 2. LOAD VALIDATED ACCOUNT-LEVEL DATASET
# ============================================================

INPUT_FILE = (
    INPUT_DIR /
    "statistical_account_level_dataset.csv"
)

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Required analytical dataset not found:\n{INPUT_FILE}"
    )

df = pd.read_csv(INPUT_FILE)

print(
    f"Accounts loaded: {len(df):,}"
)


# ============================================================
# 3. VALIDATE REQUIRED FIELDS
# ============================================================

required_columns = [
    "account_id",
    "recovered_flag",
    "total_payment_amount",
    "total_calls",
    "dpd",
    "outstanding_amount"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Required columns validated.")


# ============================================================
# 4. STANDARDIZE NUMERIC FIELDS
# ============================================================

numeric_columns = [
    "recovered_flag",
    "total_payment_amount",
    "total_calls",
    "dpd",
    "outstanding_amount"
]

for column in numeric_columns:

    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

df["recovered_flag"] = (
    df["recovered_flag"]
    .fillna(0)
    .astype(int)
)

df["total_payment_amount"] = (
    df["total_payment_amount"]
    .fillna(0)
)

df["total_calls"] = (
    df["total_calls"]
    .fillna(0)
)

df["outstanding_amount"] = (
    df["outstanding_amount"]
    .fillna(0)
)


# ============================================================
# 5. BASELINE PORTFOLIO
# ============================================================

accounts = (
    df["account_id"]
    .nunique()
)

recovered_accounts = int(
    df["recovered_flag"].sum()
)

baseline_recovery_rate = (
    recovered_accounts / accounts
    if accounts > 0
    else np.nan
)

baseline_recovery_amount = (
    df["total_payment_amount"].sum()
)

baseline_outstanding_amount = (
    df["outstanding_amount"].sum()
)

unrecovered_accounts = (
    accounts - recovered_accounts
)

if recovered_accounts > 0:

    average_recovery_per_recovered_account = (
        baseline_recovery_amount
        /
        recovered_accounts
    )

else:

    average_recovery_per_recovered_account = 0


baseline_summary = pd.DataFrame([{

    "accounts":
        accounts,

    "recovered_accounts":
        recovered_accounts,

    "unrecovered_accounts":
        unrecovered_accounts,

    "baseline_recovery_rate":
        baseline_recovery_rate,

    "baseline_recovery_amount":
        baseline_recovery_amount,

    "baseline_outstanding_amount":
        baseline_outstanding_amount,

    "average_recovery_per_recovered_account":
        average_recovery_per_recovered_account

}])


print("\n" + "=" * 90)
print("BASELINE PORTFOLIO")
print("=" * 90)

display(baseline_summary)

baseline_summary.to_csv(
    OUTPUT_DIR /
    "counterfactual_baseline.csv",
    index=False
)


# ============================================================
# 6. COUNTERFACTUAL SCENARIO FUNCTION
# ============================================================
#
# We use an explicit incremental recovery-rate assumption.
#
# Example:
# A 1% absolute improvement means:
#
#       baseline rate + 0.01
#
# It is NOT interpreted as proof that the intervention
# will actually produce that improvement.
# ============================================================

def run_scenario(
    scenario_name,
    description,
    eligible_mask,
    assumed_absolute_lift
):

    eligible_accounts = int(
        eligible_mask.sum()
    )

    if eligible_accounts == 0:

        return {

            "scenario":
                scenario_name,

            "description":
                description,

            "eligible_accounts":
                0,

            "baseline_eligible_recovery_rate":
                np.nan,

            "assumed_absolute_recovery_lift":
                assumed_absolute_lift,

            "counterfactual_recovery_rate":
                np.nan,

            "estimated_incremental_recovered_accounts":
                0,

            "estimated_incremental_recovery_amount":
                0,

            "estimated_total_recovery_amount":
                baseline_recovery_amount,

            "causal_claim":
                False
        }

    eligible_baseline_rate = (
        df.loc[
            eligible_mask,
            "recovered_flag"
        ].mean()
    )

    counterfactual_rate = min(
        eligible_baseline_rate
        +
        assumed_absolute_lift,
        1.0
    )

    incremental_rate = (
        counterfactual_rate
        -
        eligible_baseline_rate
    )

    incremental_recovered_accounts = (
        eligible_accounts
        *
        incremental_rate
    )

    incremental_recovery_amount = (
        incremental_recovered_accounts
        *
        average_recovery_per_recovered_account
    )

    estimated_total_recovery_amount = (
        baseline_recovery_amount
        +
        incremental_recovery_amount
    )

    return {

        "scenario":
            scenario_name,

        "description":
            description,

        "eligible_accounts":
            eligible_accounts,

        "baseline_eligible_recovery_rate":
            eligible_baseline_rate,

        "assumed_absolute_recovery_lift":
            assumed_absolute_lift,

        "counterfactual_recovery_rate":
            counterfactual_rate,

        "estimated_incremental_recovered_accounts":
            incremental_recovered_accounts,

        "estimated_incremental_recovery_amount":
            incremental_recovery_amount,

        "estimated_total_recovery_amount":
            estimated_total_recovery_amount,

        "causal_claim":
            False
    }


# ============================================================
# 7. SCENARIO A
# ============================================================
#
# Portfolio-wide conservative improvement.
#
# Assumption:
# +1 percentage point recovery-rate improvement among
# currently unrecovered accounts.
# ============================================================

scenario_a_mask = (
    df["recovered_flag"] == 0
)

scenario_a = run_scenario(

    scenario_name="A",

    description=(
        "1 percentage-point recovery improvement "
        "among currently unrecovered accounts"
    ),

    eligible_mask=scenario_a_mask,

    assumed_absolute_lift=0.01
)


# ============================================================
# 8. SCENARIO B
# ============================================================
#
# Low-exposure opportunity.
#
# Stage 5 showed that recovery rates were broadly similar
# across call-exposure groups, so this is NOT treated as proof
# that more calls cause recovery.
#
# We therefore model this only as a planning scenario.
#
# Assumption:
# +1 percentage point recovery improvement among currently
# unrecovered accounts with <=2 calls.
# ============================================================

scenario_b_mask = (
    (df["recovered_flag"] == 0)
    &
    (df["total_calls"] <= 2)
)

scenario_b = run_scenario(

    scenario_name="B",

    description=(
        "1 percentage-point recovery improvement "
        "among currently unrecovered accounts with <=2 calls"
    ),

    eligible_mask=scenario_b_mask,

    assumed_absolute_lift=0.01
)


# ============================================================
# 9. SCENARIO C
# ============================================================
#
# Moderate sensitivity scenario.
#
# Assumption:
# +2 percentage points among currently unrecovered accounts.
# ============================================================

scenario_c_mask = (
    df["recovered_flag"] == 0
)

scenario_c = run_scenario(

    scenario_name="C",

    description=(
        "2 percentage-point recovery improvement "
        "among currently unrecovered accounts"
    ),

    eligible_mask=scenario_c_mask,

    assumed_absolute_lift=0.02
)


# ============================================================
# 10. COMBINE SCENARIOS
# ============================================================

scenario_df = pd.DataFrame([
    scenario_a,
    scenario_b,
    scenario_c
])

print("\n" + "=" * 90)
print("COUNTERFACTUAL SCENARIOS")
print("=" * 90)

display(scenario_df)

scenario_df.to_csv(
    OUTPUT_DIR /
    "counterfactual_scenarios.csv",
    index=False
)


# ============================================================
# 11. SENSITIVITY ANALYSIS
# ============================================================
#
# Instead of assuming one intervention effect, test a range:
#
# 0.5 pp, 1 pp, 1.5 pp, 2 pp, 3 pp
#
# This shows how sensitive the estimated value is to the
# assumed improvement.
# ============================================================

print("\n" + "=" * 90)
print("COUNTERFACTUAL SENSITIVITY ANALYSIS")
print("=" * 90)

sensitivity_lifts = [
    0.005,
    0.010,
    0.015,
    0.020,
    0.030
]

sensitivity_rows = []

eligible_accounts = (
    unrecovered_accounts
)

for lift in sensitivity_lifts:

    incremental_recovered_accounts = (
        eligible_accounts
        *
        lift
    )

    incremental_recovery_amount = (
        incremental_recovered_accounts
        *
        average_recovery_per_recovered_account
    )

    estimated_total_recovery_amount = (
        baseline_recovery_amount
        +
        incremental_recovery_amount
    )

    sensitivity_rows.append({

        "assumed_absolute_recovery_lift":
            lift,

        "eligible_unrecovered_accounts":
            eligible_accounts,

        "estimated_incremental_recovered_accounts":
            incremental_recovered_accounts,

        "estimated_incremental_recovery_amount":
            incremental_recovery_amount,

        "estimated_total_recovery_amount":
            estimated_total_recovery_amount,

        "causal_claim":
            False
    })


sensitivity_df = pd.DataFrame(
    sensitivity_rows
)

display(sensitivity_df)

sensitivity_df.to_csv(
    OUTPUT_DIR /
    "counterfactual_sensitivity.csv",
    index=False
)


# ============================================================
# 12. LOW-EXPOSURE OPPORTUNITY
# ============================================================

low_exposure = df[
    df["total_calls"] <= 2
].copy()

low_exposure_accounts = (
    low_exposure["account_id"]
    .nunique()
)

low_exposure_recovered = int(
    low_exposure["recovered_flag"]
    .sum()
)

low_exposure_rate = (
    low_exposure_recovered
    /
    low_exposure_accounts
    if low_exposure_accounts > 0
    else np.nan
)

low_exposure_recovery_amount = (
    low_exposure["total_payment_amount"]
    .sum()
)

low_exposure_summary = pd.DataFrame([{

    "definition":
        "Accounts with <=2 calls",

    "accounts":
        low_exposure_accounts,

    "recovered_accounts":
        low_exposure_recovered,

    "recovery_rate":
        low_exposure_rate,

    "recovery_amount":
        low_exposure_recovery_amount

}])


print("\n" + "=" * 90)
print("LOW-EXPOSURE OPPORTUNITY")
print("=" * 90)

display(low_exposure_summary)

low_exposure_summary.to_csv(
    OUTPUT_DIR /
    "counterfactual_low_exposure.csv",
    index=False
)


# ============================================================
# 13. ASSUMPTION REGISTER
# ============================================================

assumption_register = pd.DataFrame([

    {
        "assumption":
            "Scenario A lift",

        "value":
            "1 percentage point",

        "interpretation":
            "Planning assumption, not causal estimate"
    },

    {
        "assumption":
            "Scenario B lift",

        "value":
            "1 percentage point",

        "interpretation":
            "Planning assumption, not causal estimate"
    },

    {
        "assumption":
            "Scenario C lift",

        "value":
            "2 percentage points",

        "interpretation":
            "Planning assumption, not causal estimate"
    },

    {
        "assumption":
            "Sensitivity range",

        "value":
            "0.5 to 3 percentage points",

        "interpretation":
            "Tests business-case sensitivity"
    },

    {
        "assumption":
            "Recovery value",

        "value":
            "Historical average recovery per recovered account",

        "interpretation":
            "Used only for scenario sizing"
    },

    {
        "assumption":
            "Causal interpretation",

        "value":
            "Not claimed",

        "interpretation":
            "Scenarios represent planning estimates"
    },

    {
        "assumption":
            "Raw source data",

        "value":
            "Not modified",

        "interpretation":
            "Data governance requirement"
    }
])

print("\n" + "=" * 90)
print("COUNTERFACTUAL ASSUMPTION REGISTER")
print("=" * 90)

display(assumption_register)

assumption_register.to_csv(
    OUTPUT_DIR /
    "counterfactual_assumptions.csv",
    index=False
)


# ============================================================
# 14. BEST-CASE SCENARIO SUMMARY
# ============================================================

best_index = (
    scenario_df[
        "estimated_incremental_recovery_amount"
    ]
    .idxmax()
)

best_scenario = (
    scenario_df.loc[best_index]
)


final_summary = pd.DataFrame([{

    "baseline_accounts":
        accounts,

    "baseline_recovered_accounts":
        recovered_accounts,

    "baseline_recovery_rate":
        baseline_recovery_rate,

    "baseline_recovery_amount":
        baseline_recovery_amount,

    "largest_scenario":
        best_scenario["scenario"],

    "largest_scenario_incremental_recovery_amount":
        best_scenario[
            "estimated_incremental_recovery_amount"
        ],

    "counterfactual_is_causal_claim":
        False

}])


print("\n" + "=" * 90)
print("COUNTERFACTUAL ANALYSIS SUMMARY")
print("=" * 90)

display(final_summary)

final_summary.to_csv(
    OUTPUT_DIR /
    "counterfactual_analysis_summary.csv",
    index=False
)


# ============================================================
# 15. SAVE ANALYTICAL INPUT COPY
# ============================================================

df.to_csv(
    OUTPUT_DIR /
    "counterfactual_account_level_dataset.csv",
    index=False
)


# ============================================================
# 16. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("COUNTERFACTUAL ANALYSIS COMPLETE")
print("=" * 90)

print(
    "Validated account-level analytical data was used."
)

print(
    "Raw source files were NOT modified."
)

print(
    "Scenario improvements are explicit assumptions."
)

print(
    "Counterfactual estimates are NOT causal effects."
)

print(
    f"Outputs saved to: {OUTPUT_DIR}"
)

CREDRESOLVE — COUNTERFACTUAL ANALYSIS
Accounts loaded: 30,000
Required columns validated.

BASELINE PORTFOLIO


,accounts,recovered_accounts,unrecovered_accounts,baseline_recovery_rate,baseline_recovery_amount,baseline_outstanding_amount,average_recovery_per_recovered_account
0,30000,13284,16716,0.4428,1.917259e+09,1.048904e+10,144328.411408



COUNTERFACTUAL SCENARIOS


,scenario,description,eligible_accounts,baseline_eligible_recovery_rate,assumed_absolute_recovery_lift,counterfactual_recovery_rate,estimated_incremental_recovered_accounts,estimated_incremental_recovery_amount,estimated_total_recovery_amount,causal_claim
0,A,1 percentage-point recovery improvement among ...,16716,0.0,0.01,0.01,167.16,2.412594e+07,1.941385e+09,False
1,B,1 percentage-point recovery improvement among ...,7090,0.0,0.01,0.01,70.90,1.023288e+07,1.927492e+09,False
2,C,2 percentage-point recovery improvement among ...,16716,0.0,0.02,0.02,334.32,4.825187e+07,1.965510e+09,False



COUNTERFACTUAL SENSITIVITY ANALYSIS


...


COUNTERFACTUAL ASSUMPTION REGISTER


,assumption,value,interpretation
0,Scenario A lift,1 percentage point,"Planning assumption, not causal estimate"
1,Scenario B lift,1 percentage point,"Planning assumption, not causal estimate"
2,Scenario C lift,2 percentage points,"Planning assumption, not causal estimate"
3,Sensitivity range,0.5 to 3 percentage points,Tests business-case sensitivity
4,Recovery value,Historical average recovery per recovered account,Used only for scenario sizing
5,Causal interpretation,Not claimed,Scenarios represent planning estimates
6,Raw source data,Not modified,Data governance requirement



COUNTERFACTUAL ANALYSIS SUMMARY


,baseline_accounts,baseline_recovered_accounts,baseline_recovery_rate,baseline_recovery_amount,largest_scenario,largest_scenario_incremental_recovery_amount,counterfactual_is_causal_claim
0,30000,13284,0.4428,1.917259e+09,C,4.825187e+07,False



COUNTERFACTUAL ANALYSIS COMPLETE
Validated account-level analytical data was used.
Raw source files were NOT modified.
Scenario improvements are explicit assumptions.
Counterfactual estimates are NOT causal effects.
Outputs saved to: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
